In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.6/219.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: setuptools
    Found existing 

In [2]:
import numpy as np
from scipy.stats import norm

def bs_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_vega(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T)

def newton_iv(market_price, S, K, T, r, option_type='call', guess=0.3, tol=1e-6, max_iter=100):
    sigma = max(guess, 0.01)
    for i in range(max_iter):
        if option_type == 'call':
            price = bs_call_price(S, K, T, r, sigma)
        else:
            price = bs_put_price(S, K, T, r, sigma)
        diff = price - market_price
        if abs(diff) < tol:
            return round(sigma, 6)
        vega = bs_vega(S, K, T, r, sigma)
        if vega < 1e-12:
            break
        sigma = sigma - diff / vega
        sigma = max(sigma, 0.001)
    return None

S = 64000
K = 65000
T = 14 / 365
r = 0.045
true_iv = 0.75
market_call = bs_call_price(S, K, T, r, true_iv)
print('BTC当前价: $', S)
print('行权价: $', K)
print('到期: 14天')
print('市场看涨期权价: $', round(market_call, 2))
iv = newton_iv(market_call, S, K, T, r, 'call')
print('反解IV: ', round(iv*100, 2), '%')

S2 = 7457
K2 = 7500
T2 = 30 / 365
market_call2 = bs_call_price(S2, K2, T2, r, 0.20)
print()
print('标普500对比')
print('SPX当前价: $', S2)
print('市场看涨期权价: $', round(market_call2, 2))
iv2 = newton_iv(market_call2, S2, K2, T2, r, 'call')
print('反解IV: ', round(iv2*100, 2), '%')

market_put = bs_put_price(S, K, T, r, true_iv)
print()
print('BTC看跌期权')
print('市场看跌期权价: $', round(market_put, 2))
iv_put = newton_iv(market_put, S, K, T, r, 'put')
print('反解IV: ', round(iv_put*100, 2), '%')

print()
print('不同到期日对比')
for days in [7, 14, 30, 60]:
    Ti = days / 365
    mp = bs_call_price(S, K, Ti, r, true_iv)
    ivi = newton_iv(mp, S, K, Ti, r, 'call')
    print('到期', days, '天 价=$', round(mp, 2), ' IV=', round(ivi*100, 2), '%')


BTC当前价: $ 64000
行权价: $ 65000
到期: 14天
市场看涨期权价: $ 3345.55
反解IV:  75.0 %

标普500对比
SPX当前价: $ 7457
市场看涨期权价: $ 163.18
反解IV:  20.0 %

BTC看跌期权
市场看跌期权价: $ 4233.46
反解IV:  75.0 %

不同到期日对比
到期 7 天 价=$ 2224.73  IV= 75.0 %
到期 14 天 价=$ 3345.55  IV= 75.0 %
到期 30 天 价=$ 5140.15  IV= 75.0 %
到期 60 天 价=$ 7507.84  IV= 75.0 %
